# Frequentist Sensitivity Estimate (N-D)

This notebook demonstrates the N-dimensional Feldman-Cousins optimization workflow with `SNSensitivityEstimate`.

Workflow overview:
1. Load ROOT datasets and metadata.
2. Build `DataProcessND` objects for a multivariate search.
3. Evaluate a trial ROI by hand.
4. Optimize ROI boundaries in index space with `Metaheuristics.jl`.
5. Decode and report the best physical ROI and sensitivity.

In [1]:
import Pkg;
Pkg.activate("/sps/nemo/scratch/mpetro/Sensitivity_training/")
using SNSensitivityEstimate, UnROOT, FHist, CairoMakie, CSV, DataFramesMeta, Metaheuristics
using Distributions

  Activating project at `/sps/nemo/scratch/mpetro/Sensitivity_training`


## Step 1: Load data

In [2]:
data_info = CSV.read("/sps/nemo/scratch/mpetro/Sensitivity_training/data/data_info.csv", DataFrame)
file_paths = joinpath.("/sps/nemo/scratch/mpetro/Sensitivity_training/data", data_info.file_name)
data_files = [UnROOT.ROOTFile(file_path) for file_path in file_paths]
tree_name = "tree"
variables = keys(data_files[1][tree_name])
data_tables = [UnROOT.LazyTree(data_file, tree_name, variables) for data_file in data_files]

5-element Vector{LazyTree with 26 branches:
sameSide, reconstructedEnergy2, Pext, avgE, z2Reconstructed...
}:
 ┌─────┬──────────┬─────────────────┬────────────┬───────────┬───────────────────
│ Row │ sameSide │ reconstructedEn │ Pext       │ avgE      │ z2Reconstructed  ⋯
│     │ Float32  │ Float32         │ Float32    │ Float32   │ Float32          ⋯
├─────┼──────────┼─────────────────┼────────────┼───────────┼───────────────────
│ 1   │ 0.0      │ 417.84256       │ 2.062762e- │ 539.3591  │ 36.14118         ⋯
│ 2   │ 1.0      │ 66.81933        │ 0.4553107  │ 872.25037 │ -62.85587        ⋯
│ 3   │ 1.0      │ 286.42862       │ 7.6636536e │ 1108.881  │ 796.15796        ⋯
│ 4   │ 0.0      │ 163.08542       │ 0.00024017 │ 701.16925 │ -1109.882        ⋯
│ 5   │ 0.0      │ 420.899         │ 5.03516e-1 │ 474.4948  │ -773.49896       ⋯
│ 6   │ 0.0      │ 771.45776       │ 3.876098e- │ 1004.4069 │ 591.2683         ⋯
│ 7   │ 0.0      │ 1874.1747       │ 1.8641002e │ 1229.9344 │ 57.884754        

## Step 2: Define multivariate search space

`var_bounds` defines the allowed physical domain for each variable.
`var_steps` sets grid resolution for each edge in the optimizer index space.

You can tune `var_steps` to trade off speed and precision.

In [5]:
var_names = [
    "sumE",
    "phi",
    "dy",
    "dz",
    "lPint",
    "lPext",
]

var_bounds = (
    sumE = (300, 3500),
    phi = (0,180),
    dy = (0, 200),
    dz = (0, 200),
    lPint = (0.0, 10.0),
    lPext = (0.0, 110.0),
)

# Default search resolution (adjust if needed)
var_steps = (
    sumE = 100.0, 
    phi = 5.0,
    dy = 5.0,
    dz = 5.0,
    lPint = 0.01,
    lPext = 0.05,
)

α = 1.64485362695147

selected_tables = [UnROOT.LazyTree(data_file, tree_name, var_names) for data_file in data_files]

5-element Vector{LazyTree with 6 branches:
dz, lPext, sumE, phi, dy...
}:
 ┌─────┬───────────┬────────────┬───────────┬───────────┬───────────┬────────────
│ Row │ dz        │ lPext      │ sumE      │ phi       │ dy        │ lPint     ⋯
│     │ Float32   │ Float32    │ Float32   │ Float32   │ Float32   │ Float32   ⋯
├─────┼───────────┼────────────┼───────────┼───────────┼───────────┼────────────
│ 1   │ 5.6566067 │ 4.6855507  │ 1078.7181 │ 165.64183 │ 19.030518 │ 1.1110213 ⋯
│ 2   │ 154.93277 │ 0.34169215 │ 1744.5007 │ 88.365906 │ 78.09949  │ 1.8118192 ⋯
│ 3   │ 238.08252 │ 6.1155643  │ 2217.762  │ 70.08641  │ 164.70947 │ 0.2806641 ⋯
│ 4   │ 8.0251465 │ 3.6194775  │ 1402.3385 │ 124.14437 │ 10.782349 │ 1.3537722 ⋯
│ 5   │ 39.54541  │ 12.297987  │ 948.9896  │ 133.5398  │ 1.5751953 │ 0.4806245 ⋯
│ 6   │ 23.649902 │ 11.411605  │ 2008.8138 │ 155.11577 │ 14.871338 │ 0.5717010 ⋯
│ 7   │ 1.7066956 │ 21.72953   │ 2459.869  │ 112.62566 │ 2.8797607 │ 0.1793010 ⋯
│ 8   │ 6.4183044 │ 12.672332  │ 2

## Step 3: Build ND process objects and split signal/background

In [13]:
processes_nd = [
    SNSensitivityEstimate.DataProcessND(
        selected_tables[i],
        String(data_info.process[i]),
        data_info.is_signal[i],
        data_info.activity[i],
        data_info.time_s[i],
        data_info.n_sim[i],
        var_bounds,
        data_info.amount[i],
        var_names,
    ) for i in 1:length(selected_tables)
]

signal_name = "bb0nu_foil_bulk"
signal_process = get_process(signal_name, processes_nd)[1]

background_names = data_info.process[data_info.process .!= signal_name]
background_processes = [SNSensitivityEstimate.get_process(String(name), processes_nd)[1] for name in background_names]

creating process: Bi214_foil_bulk
creating process: Bi214_wire_surface
creating process: Tl208_foil_bulk
creating process: bb0nu_foil_bulk
creating process: bb_foil_bulk


4-element Vector{DataProcessND{Float64, Bool, @NamedTuple{sumE::Tuple{Int64, Int64}, phi::Tuple{Int64, Int64}, dy::Tuple{Int64, Int64}, dz::Tuple{Int64, Int64}, lPint::Tuple{Float64, Float64}, lPext::Tuple{Float64, Float64}}, String, Int64}}:
 DataProcessND{Float64, Bool, @NamedTuple{sumE::Tuple{Int64, Int64}, phi::Tuple{Int64, Int64}, dy::Tuple{Int64, Int64}, dz::Tuple{Int64, Int64}, lPint::Tuple{Float64, Float64}, lPext::Tuple{Float64, Float64}}, String, Int64}(UnROOT.LazyEvent[UnROOT.LazyEvent at index 1 with 6 columns:
(dz = 5.6566067f0, lPext = 4.6855507f0, sumE = 1078.7181f0, phi = 165.64183f0, dy = 19.030518f0, lPint = 1.1110213f0), UnROOT.LazyEvent at index 2 with 6 columns:
(dz = 154.93277f0, lPext = 0.34169215f0, sumE = 1744.5007f0, phi = 88.365906f0, dy = 78.09949f0, lPint = 1.8118192f0), UnROOT.LazyEvent at index 3 with 6 columns:
(dz = 238.08252f0, lPext = 6.1155643f0, sumE = 2217.762f0, phi = 70.08641f0, dy = 164.70947f0, lPint = 0.28066418f0), UnROOT.LazyEvent at index 4

## Step 4: Evaluate a trial ROI

ROI must define a `(min, max)` window for every variable in `var_names`.

In [ ]:
my_roi = (
    sumE = (2000, 3100),
    phi = (0, 180),
    dy = (0, 100),
    dz = (0, 100),
    lPint = (0.0, 1.0),
    lPext = (0.1, 110.0),
)

my_sensitivity = get_sensitivityND(SNparams, α, processes_nd, my_roi; approximate = "table")
print(my_sensitivity)

tHalf: 2.1576491266720884e23
signalEff: 0.2359678
bkg count: 2926.7528492857755
best ROI:
  sumE : (2000, 3100)
  phi : (0, 180)
  dy : (0, 100)
  dz : (0, 100)
  lPint : (0.0, 1.0)
  lPext : (0.1, 110.0)


## Step 5: Build optimization problem

The optimizer works in index space over ROI edge coordinates.
We optimize `-S/B` using the faster `formula` approximation during search.

In [8]:
grid = GridSpec(signal_process, var_steps)

prob(proposed_indices) = -get_s_to_b(SNparams, α, processes_nd, proposed_indices, grid; approximate = "formula")

searchRange = make_stepRange(grid)
lower_bound = [x[1] for x in searchRange] .|> float
upper_bound = [x[2] for x in searchRange] .|> float

options = Options(;
    time_limit = 10 * 20.0, # optimization time limit
    verbose = true,
)

bounds = boxconstraints(lb = lower_bound, ub = upper_bound)

BoxConstrainedSpace{Float64}([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [32.0, 32.0, 36.0, 36.0, 40.0, 40.0, 40.0, 40.0, 1000.0, 1000.0, 2200.0, 2200.0], [32.0, 32.0, 36.0, 36.0, 40.0, 40.0, 40.0, 40.0, 1000.0, 1000.0, 2200.0, 2200.0], 12, true)

## Step 6: Seed an initial guess and optimize

`x0` is given in index units and follows the order of `var_names`:
`sumE_min, sumE_max, dy_min, dy_max, ...`.

In [11]:
x0 = float.([
    value_to_index(2700, grid.mins[1], grid.steps[1]; nsteps = grid.nsteps[1]), value_to_index(3100, grid.mins[1], grid.steps[1]; nsteps = grid.nsteps[1]),
    value_to_index(0, grid.mins[2], grid.steps[2]; nsteps = grid.nsteps[2]), value_to_index(180, grid.mins[2], grid.steps[2]; nsteps = grid.nsteps[2]),
    value_to_index(0, grid.mins[3], grid.steps[3]; nsteps = grid.nsteps[3]), value_to_index(100, grid.mins[3], grid.steps[3]; nsteps = grid.nsteps[3]),
    value_to_index(0, grid.mins[4], grid.steps[4]; nsteps = grid.nsteps[4]), value_to_index(140, grid.mins[4], grid.steps[4]; nsteps = grid.nsteps[4]),
    value_to_index(0.0, grid.mins[5], grid.steps[5]; nsteps = grid.nsteps[5]), value_to_index(4.0, grid.mins[5], grid.steps[5]; nsteps = grid.nsteps[5]),
    value_to_index(2.0, grid.mins[6], grid.steps[6]; nsteps = grid.nsteps[6]), value_to_index(100.0, grid.mins[6], grid.steps[6]; nsteps = grid.nsteps[6]),
])

algo = PSO(; options)
set_user_solutions!(algo, x0, prob)

result = optimize(prob, bounds, algo)
@show minimum(result)
@show res = minimizer(result)

+-----------+------------+------------+------------+------------+
| Iteration | Num. Evals |   Minimum  |    Time    | Converged  |
+-----------+------------+------------+------------+------------+
|         1 |        120 | -5.4848e-02 |   3.6536 s |         No | 
|         2 |        239 | -5.5351e-02 | 241.5288 s |         No | 
minimum(result) = -0.05535147413698321
res = minimizer(result) = [23.63991308137453, 28.4323749704908, 0.0, 36.0, 0.0, 20.05026970136462, 0.0, 29.177466696712344, 0.0, 399.4074922405942, 0.0, 2132.025887755289]


12-element Vector{Float64}:
   23.63991308137453
   28.4323749704908
    0.0
   36.0
    0.0
   20.05026970136462
    0.0
   29.177466696712344
    0.0
  399.4074922405942
    0.0
 2132.025887755289

## Step 7: Decode best ROI and compute precise sensitivity

In [22]:
best = get_best_ROI_ND(res, signal_process, grid)
best_sens = get_sensitivityND(SNparams, α, processes_nd, best; approximate = "table")

println("\n===== BEST RESULT =====")
print(best_sens)


===== BEST RESULT =====
tHalf: 4.798961847593858e24
signalEff: 0.1964976
bkg count: 1.5307282009957193
best ROI:
  sumE : (2700.0, 3100.0)
  phi : (0.0, 180.0)
  dy : (0.0, 100.0)
  dz : (0.0, 145.0)
  lPint : (0.0, 3.99)
  lPext : (0.0, 106.60000000000001)


# STEP 8: plot results

In [ ]:
let
    binning = 0:100:3500
    plot_var = :sumE
    best_ROI = best
    min_a, min_b = best_ROI[plot_var]
    best_ROI = merge(best_ROI, (sumE = (binning[1], binning[end]),))
    best_sens = get_sensitivityND(SNparams, α, processes_nd, best; approximate = "table").tHalf



    # calculate signal activity for the optimized sensitivity and ROI
    best_0nu_activity = SNSensitivityEstimate.halfLife_to_activity(SNparams["Nₐ"], SNparams["W"], best_sens*365*24*3600)
    SNSensitivityEstimate.set_activity!(signal_process, best_0nu_activity)

    bkg_histograms = [SNSensitivityEstimate.get_roi_bkg_counts_hist(p, best_ROI, binning, plot_var) for p in background_processes]
    signal_histogram = SNSensitivityEstimate.get_roi_bkg_counts_hist(signal_process, best_ROI, binning, plot_var)
    colors = ["#041E42", "#BE4D00", "#951272", "#006630", "#005C8A", "#FFB948", "#605643", "#302D23"]
    f = CairoMakie.Figure(size = (1800, 800), fontsize = 24)
    a1 = CairoMakie.Axis(f[1, 1], title = "Background model with best signal overlay", xlabel = "sum of energies", ylabel = "expected counts", yscale = log10)
    a2 = CairoMakie.Axis(f[1, 2], title = "Zoom to best ROI", xlabel = "sum of energies", ylabel = "expected counts")

    for axis in (a1, a2)
        for i in length(bkg_histograms):-1:1
            CairoMakie.stephist!(axis, sum(bkg_histograms[1:i]), color = colors[i], linewidth = 5)
        end
        CairoMakie.stephist!(axis, signal_histogram, color = :black, linewidth = 4, linestyle = :dash)
        CairoMakie.vspan!(axis, min_a, min_b, color = (:darkgreen, 0.10))
    end

    labels = [String(p.isotopeName) for p in background_processes]
    push!(labels, "signal")

    elements = [CairoMakie.LineElement(color = colors[i], linewidth = 5) for i in 1:length(background_processes)]
    push!(elements, CairoMakie.LineElement(color = :black, linestyle = :dash, linewidth = 5))

    ylims!(a1, 1e-3, 1e5)
    ylims!(a2, 0, 5)
    xlims!(a2, min_a, min_b)
    CairoMakie.textlabel!(
        a2,
        min_a,
        4.6,
        text = "T_1/2 = $(round(best_sens, digits=3)) yr",
        text_align = (:left, :top),
        offset = (100, 0),
        text_color = :black,
        fontsize = 24,
        background_color = (:white, 0.92),
        strokecolor = :darkgreen,
        strokewidth = 4,
        cornerradius = 8,
        padding = (10, 8, 10, 8),
    )

    CairoMakie.Legend(f[1,3], elements, labels, "Processes", patchsize = (45, 20))
    save("/sps/nemo/scratch/mpetro/Sensitivity_training/data/out/ex4/best_roi_plot.png", f, px_per_unit = 2)
    f
end

